### Imports

In [1]:
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
import os
import torchvision.transforms as transforms
import torch.nn.functional as F
import lpips
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import cv2
import glob
import subprocess
import shutil
from tqdm import tqdm
from PIL import ImageFilter
from torch.utils.data import random_split

/medias/db/ImagingSecurity_misc/adra/e2vid/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from retinaface import RetinaFace

In [ ]:
import numba
import tensorflow as tf
import keras 

# Synthesize Event Data

Setup V2E event simulator from the following GitHub repo: https://github.com/SensorsINI/v2e

and then open folder using: 

In [ ]:
!cd /your_directory/v2e

Simulate your dataset using "Noisy" or "Clean" V2E configurations. 

This function operates by folder. 
If your dataset is made up of class folders, run each class seperately.

In [ ]:
# Modify this function to handle processing your dataset structure
def process_videos(data_root, output_root):
    # Recursively find all .avi video files in the dataset
    valid_videos = sorted(glob.glob(os.path.join(data_root, "*.avi")))

    # Loop through each video file and apply the v2e conversion
    for video_path in valid_videos:
        # Extract the class folder (e.g., 'class1') and video name
        class_folder = os.path.basename(os.path.dirname(video_path))
        video_name = os.path.splitext(os.path.basename(video_path))[0]

        # Define output folder structure
        out_folder = os.path.join(output_root, class_folder, video_name)

        if not os.path.exists(out_folder):
            os.makedirs(out_folder)

        # Build the v2e command using your specific parameters
        v2e_command_clean = [
            "v2e.py",
            "-i", video_path,
            "-o", out_folder,
            "--overwrite",
            "--unique_output_folder", "false",
            "--dvs_h5", "None",
            "--dvs_aedat2", "None",
            "--dvs_text", "dvs_events.txt",
            "--no_preview",
            "--dvs_exposure", "duration", ".033",
            "--input_frame_rate", "30",
            "--input_slowmotion_factor", "1",
            "--disable_slomo",
            "--auto_timestamp_resolution", "false",
            "--pos_thres", "0.2",
            "--neg_thres", "0.2",
            "--sigma_thres", "0.02",
            "--cutoff_hz", "0",
            "--leak_rate_hz", "0",
            "--shot_noise_rate_hz", "0",
            "--dvs346"
        ]

        v2e_command_noisy = [
            "v2e.py",
            "-i", video_path,
            "-o", out_folder,
            "--overwrite",
            "--unique_output_folder", "false",
            "--dvs_h5", "None",
            "--dvs_aedat2", "None",
            "--dvs_text", "dvs_events.txt",
            "--no_preview",
            "--dvs_exposure", "duration", ".033",
            "--input_frame_rate", "30",
            "--input_slowmotion_factor", "1",
            "--disable_slomo",
            "--auto_timestamp_resolution", "false",
            "--pos_thres", "0.2",
            "--neg_thres", "0.2",
            "--sigma_thres", "0.03",
            "--cutoff_hz", "30",
            "--leak_rate_hz", "0.1",
            "--shot_noise_rate_hz", "5",
            "--dvs346"
        ]


        # Execute the v2e command for each video
        subprocess.run(v2e_command_noisy)


In [ ]:
data_root = "dataset_directory/class"
output_root = "destination_dataset/"

process_videos(data_root, output_root)

V2E generates a Video and Text file per simulation. This function deletes the video and keeps the .txt files.

In [ ]:

def post_process_output(output_root):
    # Traverse the output_root directory
    for root, dirs, files in os.walk(output_root):
        for dir_name in dirs:
            video_folder_path = os.path.join(root, dir_name)

            # Ignore the root-level folder itself
            if root == output_root:
                continue

            video_name = dir_name  # Video name matches the directory name

            # Define paths for the dvs_video.avi and dvs_events.txt
            video_file_path = os.path.join(video_folder_path, "dvs-video.avi")
            events_file_path = os.path.join(video_folder_path, "dvs_events.txt")

            # Define the new path for the renamed events file
            new_events_file_path = os.path.join(output_root, f"{video_name}.txt")

            # Delete the dvs_video.avi file
            if os.path.exists(video_file_path):
                os.remove(video_file_path)
                print(f"Deleted video file: {video_file_path}")

            # Move and rename dvs_events.txt to output root and rename it to Video1_name.txt
            if os.path.exists(events_file_path):
                shutil.move(events_file_path, new_events_file_path)
                print(f"Moved and renamed events file: {events_file_path} to {new_events_file_path}")

            # Remove the video directory now that it's empty
            if os.path.exists(video_folder_path):
                os.rmdir(video_folder_path)
                print(f"Deleted video folder: {video_folder_path}")

In [ ]:
post_process_output("destination_dataset")

# Extract RGB Frames (from original)

To synchronize with the V2E-generated event data, you need to extract frames at exact time intervals. Since each frame corresponds to 33.33 ms in your event data (with --dvs_exposure duration .033), you need to ensure that the frame extraction aligns with this interval.

In [7]:
def extract_frames_anonym(dataset_dir, output_dir='frames', width=346, height=260, fps=30):
    os.makedirs(output_dir, exist_ok=True)

    videos_to_process = []

    # Process videos directly from the dataset/videos folder (no class subfolders)
    for video_file in sorted(os.listdir(dataset_dir)):
        if video_file.endswith(('.avi', '.mp4', '.mov')):
            video_path = os.path.join(dataset_dir, video_file)
            video_name = os.path.splitext(video_file)[0]  # Extract video name without extension
            videos_to_process.append((video_path, video_name))

    # Use a ThreadPoolExecutor to process videos in parallel
    with ThreadPoolExecutor(max_workers=4) as executor:  # Adjust the number of workers as needed
        futures = [executor.submit(process_video_anonym, video[0], video[1], output_dir, width, height, fps) for video in videos_to_process]

        for future in as_completed(futures):
            try:
                future.result()
            except Exception as exc:
                print(f'Video processing generated an exception: {exc}')

def process_video_anonym(video_path, video_name, output_dir, width, height, fps=30):
    cap = cv2.VideoCapture(video_path)

    # Calculate the number of frames to skip based on the fps (33.33 ms corresponds to 1 frame at 30fps)
    frame_duration = 1 / fps  # Duration of each frame in seconds (e.g., 1/30 for 30fps)

    frame_number = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Only extract every frame that corresponds to the frame duration (33.33 ms)
        if cap.get(cv2.CAP_PROP_POS_MSEC) >= frame_number * frame_duration * 1000:
            resized_frame = cv2.resize(frame, (width, height))
            frame_filename = f'{video_name}_frame_{frame_number + 1}.jpg'
            frame_output_path = os.path.join(output_dir, frame_filename)
            cv2.imwrite(frame_output_path, resized_frame)
            frame_number += 1

    cap.release()



In [ ]:
dataset_directory = "E2PRIV/dataset/videos"
output_directory = "E2PRIV/dataset/frames"

extract_frames_anonym(dataset_directory, output_directory, width=346, height=260)

# Face Bounding Box Detection

In [9]:
def generate_face_labels(frames_folder='rgb_frames', output_folder='labels'):
    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Initialize a dictionary to store bounding boxes for each video
    video_bbox_data = {}
    count =0

    # Loop through each frame in the rgb_frames folder
    for frame_file in os.listdir(frames_folder):
        if frame_file.endswith(('.jpg', '.jpeg', '.png')):
            # Full path of the image
            frame_path = os.path.join(frames_folder, frame_file)

            # Parse the video name and frame ID from the filename
            frame_name = os.path.splitext(frame_file)[0]
            #video_name, frame_id = frame_name.rsplit('_', 1)  # Assuming filenames are like videoname_frame_1.jpg
            #frame_id = int(frame_id)  # Convert frame_id to an integer
            video_name = frame_name.rsplit('_frame', 1)[0]  # Use all parts except the last as video_name
            frame_id = frame_name.split('_')[-1]  # Use the last part as frame_id

            # Run the RetinaFace detector on the frame
            resp = RetinaFace.detect_faces(frame_path)
            count+=1
            print(count)

            # If a face is detected, extract the bounding box
            if 'face_1' in resp:
                facial_area = resp['face_1']['facial_area']  # Get the facial area (x_min, y_min, x_max, y_max)
            else:
                # If no face detected, set bounding box to (0, 0, 0, 0)
                facial_area = [0, 0, 0, 0]

            # Store the bounding box information in the dictionary
            if video_name not in video_bbox_data:
                video_bbox_data[video_name] = []  # Initialize list for the video if not present

            # Append the frame id and bounding box to the list
            video_bbox_data[video_name].append([frame_id] + facial_area)

    # After processing all frames, save the bounding box data to a single file per video
    for video_name, bbox_data in video_bbox_data.items():
        output_path = os.path.join(output_folder, f"{video_name}_bbox.txt")

        # Sort bbox data by frame_id to ensure correct order
        bbox_data.sort(key=lambda x: x[0])

        # Write the bounding box data to the .txt file
        with open(output_path, 'w') as f:
            for bbox in bbox_data:
                f.write(", ".join(map(str, bbox)) + "\n")  # Convert list to string and write

        print(f"Bounding boxes saved for {video_name} as {output_path}")

In [ ]:
frames_folder = "E2PRIV/dataset/frames"
output_folder = "E2PRIV/dataset/labels"

generate_face_labels(frames_folder, output_folder)

# E2VID Training BBOX


### Load Data

In [2]:
def load_event_data(txt_file):
    """
    Load the event data from a .txt file and convert timestamps to microseconds.
    Assumes the format of each line is: timestamp, x, y, polarity.
    """
    events = []
    with open(txt_file, 'r') as f:
        for line in f:
            if line.startswith('#'):  # Skip comment lines
                continue
            data = line.split()
            # Convert timestamp to microseconds
            timestamp, x, y, p = float(data[0]) * 1e6, int(data[1]), int(data[2]), int(data[3])
            events.append([timestamp, x, y, p])

    return np.array(events).T  # Return as transposed NumPy array for easier manipulation

def events_to_voxel_grid(event_window, num_bins=5, sensor_size=(346, 260)):
    """
    Create a voxel grid from events, with timestamps in microseconds.
    """
    if event_window.size == 0:
        print("Empty event window")
        return np.zeros((num_bins, *sensor_size))  # Return empty voxel grid if no events found

    # Duration is fixed at 33.33 ms in microseconds (33330 µs)
    duration = 33.33 * 1000  # Duration in microseconds
    time_bin = duration / num_bins  # Divide time into bins
    voxel_grid = np.zeros((num_bins, *sensor_size))  # Initialize the voxel grid

    # Loop through the events and add them to the correct time bin
    for event in event_window.T:
        t, x, y, p = event
        bin_idx = int(t / time_bin)  # Determine which bin the event belongs to

        # Clamp bin_idx to ensure it stays within the valid range [0, num_bins - 1]
        bin_idx = min(max(bin_idx, 0), num_bins - 1)

        # Populate voxel grid
        voxel_grid[bin_idx, int(x), int(y)] += 1 if p == 1 else -1

    return voxel_grid


#### EventFrameDataset Class

In [ ]:
class EventFrameDataset(Dataset):
    def __init__(self, events_dir, images_dir, bboxes_dir, transform=None, frame_duration_ms=33.33, min_box_size=10):
        self.events_dir = events_dir
        self.images_dir = images_dir
        self.bboxes_dir = bboxes_dir
        self.transform = transform
        self.frame_duration_us = frame_duration_ms * 1000  # Duration of each frame in microseconds
        self.min_box_size = min_box_size

        # Load event, image, and bounding box files
        self.event_files = sorted([os.path.join(events_dir, f) for f in os.listdir(events_dir) if f.endswith('.txt')])

        # Sample 1 frame per 10 (i.e., keep every 10th frame)
        self.image_files = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith('.jpg')])[::10]
        self.bbox_files = sorted([os.path.join(bboxes_dir, f) for f in os.listdir(bboxes_dir) if f.endswith('.txt')])

    def __len__(self):
        return len(self.image_files)  # Number of frames (reduced by sampling every 10th frame)

    def load_bboxes(self, bbox_file, frame_id):
        # Load bounding boxes for a specific frame
        bboxes = []
        with open(bbox_file, 'r') as f:
            for line in f:
                bbox_frame_id, x_min, y_min, x_max, y_max = map(int, line.split(','))
                if bbox_frame_id == frame_id:
                    bboxes.append([x_min, y_min, x_max, y_max])
        return np.array(bboxes)

    def __getitem__(self, idx):
        while True:
            # Extract frame and corresponding files
            frame_file = self.image_files[idx]
            frame_basename = os.path.basename(frame_file)
            video_name = '_'.join(frame_basename.split('_')[:-2])  # Extract video name
    
            # Load the corresponding event file and bounding box file
            event_file = os.path.join(self.events_dir, video_name + '.txt')
            bbox_file = os.path.join(self.bboxes_dir, video_name + '_bbox.txt')
    
            # Load events using the supporting function
            events = load_event_data(event_file)
    
            # Extract frame index and calculate start and end times in microseconds
            frame_idx = int(frame_basename.split('_')[-1].replace('.jpg', '')) - 1
            start_time = frame_idx * self.frame_duration_us
            end_time = (frame_idx + 1) * self.frame_duration_us
    
            # Filter events within the time window
            event_indices = (events[0] >= start_time) & (events[0] < end_time)
            event_window = events[:, event_indices]
    
            # Handle empty or short event windows
            if event_window.size == 0:
                voxel_grid = torch.zeros((5, 346, 260)).float()
            else:
                # Create voxel grid for the filtered events using the supporting function
                voxel_grid = events_to_voxel_grid(event_window)
    
            # Load the frame image and apply optional transformations
            frame = Image.open(frame_file).convert('RGB')
            if self.transform:
                frame = self.transform(frame)
    
            # Convert frame to tensor
            frame = torch.tensor(np.array(frame)).float()
            # Convert voxel_grid to tensor
            voxel_grid = torch.tensor(voxel_grid).float()
    
            # Load bounding boxes for this frame
            bboxes = self.load_bboxes(bbox_file, frame_idx + 1)
    
            # Filter valid bounding boxes
            valid_bboxes = []
            for bbox in bboxes:
                x_min, y_min, x_max, y_max = bbox
                width, height = x_max - x_min, y_max - y_min
                if width >= self.min_box_size and height >= self.min_box_size:
                    valid_bboxes.append(bbox)
                else:
                    print(f"Dropping small bbox in frame {frame_file}: size {width}x{height}")
    
            # If no valid bounding boxes, skip this frame
            if not valid_bboxes:
                idx = (idx + 1) % len(self.image_files)  # Move to the next index in a circular manner
                continue
    
            # Convert bounding boxes to tensor
            valid_bboxes = torch.from_numpy(np.array(valid_bboxes)).float()
    
    
            return voxel_grid, frame, valid_bboxes


## Initialize model

In [4]:
from model.model import E2VIDRecurrent  # Assuming you have E2VID in model/

# Configuration for the model
config = {
    'num_bins': 5,
    'skip_type': 'sum',
    'num_encoders': 3,
    'base_num_channels': 32,
    'num_residual_blocks': 2,
    'norm': 'BN',
    'use_upsample_conv': False,
    'recurrent_block_type': 'convlstm',
    'learning_rate': 0.0001,
    'batch_size': 8,
    'num_epochs': 5,
    'use_gpu': torch.cuda.is_available(),
    'gpu_id': 2
}


In [ ]:
# Setup DataLoader

event_dir = 'E2PRIV/dataset/events'
frame_dir = 'E2PRIV/dataset/frames'
bbox_dir = 'E2PRIV/dataset/labels'

train_dataset = EventFrameDataset(event_dir,frame_dir,bbox_dir)

# Total dataset length
dataset_len = len(train_dataset)

# Define split sizes
train_size = int(0.8 * dataset_len)  # 80% for training
test_size = dataset_len - train_size  # Remaining 20% for testing

# Split the dataset
train_dataset, test_dataset = random_split(train_dataset, [train_size, test_size])

# Create DataLoaders for train and test sets
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True,num_workers=4, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)  # No need to shuffle the test set

### Printing (optional for verifying the data)

In [10]:
event_data, frame_data, labels = train_dataset[33]

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def draw_bbox_on_image(image, bbox):
    """
    Draws a bounding box on the given image and displays it.

    Args:
        image (PIL.Image.Image): The image object to draw the bounding box on.
        bbox (list or tuple): A list or tuple of four elements (x_min, y_min, x_max, y_max)
                              representing the corners of the bounding box.
    """
    # Create a drawing context
    draw = ImageDraw.Draw(image)

    # Unpack bounding box coordinates
    x_min, y_min, x_max, y_max = bbox

    # Draw the bounding box (rectangle)
    draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=3)

    # Display the image using matplotlib
    plt.imshow(image)
    plt.axis("off")  # Hide axis
    plt.show()

# Example usage:
img = frame_data
bbox = [133,  44, 232, 195]  # Example bounding box (x_min, y_min, x_max, y_max)
draw_bbox_on_image(img, bbox)


## Training E2PRIV

In [ ]:
import torch
print("CUDA is available:", torch.cuda.is_available())

In [ ]:
from pytorch_msssim import SSIM

# Initialize the model
model = E2VIDRecurrent(config)

pretrained_path = 'path/to/E2VID/pretrained/model/E2VID_lightweight.pth.tar'
pretrained_weights = torch.load(pretrained_path)
model.load_state_dict(pretrained_weights['state_dict'])

if config['use_gpu']:
    device = torch.device(f'cuda:{config["gpu_id"]}')
    torch.cuda.set_device(config["gpu_id"])  # Optional, sets the current device
    model = model.to(device)  # Moves model to the specified GPU

# Define separate losses
criterion_face = lpips.LPIPS(net='vgg')  # LPIPS for perceptual similarity in the face region

criterion_background = SSIM(data_range=1.0, size_average=True, channel=1)  # Replace with SSIM


if config['use_gpu']:
    criterion_face = criterion_face.to(device)  # Move to GPU if necessary
    criterion_background = criterion_background.to(device) 

optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])


#### Training Loop

In [ ]:
import time
from tqdm import tqdm
import torch
import os
import torchvision.transforms.functional as TF
import random


# Define a transform to convert RGB to grayscale
rgb_to_grayscale = transforms.Grayscale(num_output_channels=1)

# Lists to store loss values for plotting
epoch_losses = []
epoch_face_losses = []
epoch_background_losses = []
step_times = {}  # Dictionary to track time for each step

# Directory to save checkpoints
checkpoint_dir = "E2PRIV/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

start_epoch = 0

# Training Loop
for epoch in range(config['num_epochs']):
    epoch_loss = 0.0
    epoch_face_loss = 0.0
    epoch_background_loss = 0.0
    num_batches = len(train_loader)
    total_time = 0.0  # Track the total time for the epoch

    # Initialize tqdm progress bar
    progress_bar = tqdm(train_loader, total=num_batches, desc=f"Epoch {epoch + 1}/{config['num_epochs']}")

        # Training Loop
    for i, (voxel_grid, rgb_frame, bboxes) in enumerate(progress_bar): 
        # Move data to GPU if required
        if config['use_gpu']:
            voxel_grid = voxel_grid.to(device)
            rgb_frame = rgb_frame.to(device)
            bboxes = bboxes.to(device)  # Move bounding boxes to GPU
    
        # Permute dimensions for voxel grid and RGB frame
        voxel_grid = voxel_grid.permute(0, 1, 3, 2)  # Swap height and width
        rgb_frame = rgb_frame.permute(0, 3, 1, 2)  # Convert to (batch, channels, height, width)
    
        # Convert RGB to grayscale
        rgb_frame = rgb_to_grayscale(rgb_frame) / 255.0  # Normalize to [0, 1]

    
        # Apply Gaussian blur dynamically based on bounding boxes
        blurred_frame = rgb_frame.clone()  # Clone the unblurred frames to apply blur
        for idx in range(rgb_frame.size(0)):  # Loop over the batch
            for box in bboxes[idx]:  # Handle multiple bounding boxes per frame
                x_min, y_min, x_max, y_max = box.int()  # Convert bbox coordinates to int
                
                # Extract the region to blur
                face_region = blurred_frame[idx, :, y_min:y_max, x_min:x_max]
    
                # Apply Gaussian blur
                blurred_face_region = TF.gaussian_blur(face_region, kernel_size=random.choice([5, 15, 31]), sigma=random.uniform(3.0, 5.0))
                
                # Add mild Gaussian noise
                noise = torch.normal(mean=0, std=0.05, size=blurred_face_region.shape).to(rgb_frame.device)
                blurred_face_with_noise = blurred_face_region + noise
                
                # Clamp values to ensure they stay in the valid range [0, 1]
                blurred_face_with_noise = torch.clamp(blurred_face_with_noise, 0, 1)
                
                # Replace the face region with the blurred + noisy version
                blurred_frame[idx, :, y_min:y_max, x_min:x_max] = blurred_face_with_noise

        
    
        # Use blurred_frame for loss calculation
        # Forward pass
        output, _ = model(voxel_grid, None)
    
        # Normalize output and blurred frame to [-1, 1]
        output = 2 * (output - 0.5)
        blurred_frame = 2 * (blurred_frame - 0.5)

        # Compute losses (face and background as described earlier)
        face_loss = 0.0
        background_loss = 0.0
        num_face_pixels = 0
        num_background_pixels = 0
    
        for idx in range(rgb_frame.size(0)):  # Iterate over the batch
            for box in bboxes[idx]:
                x_min, y_min, x_max, y_max = box.int()
                
                # Face region for loss
                face_region_output = output[idx, :, y_min:y_max, x_min:x_max]
                face_region_target = blurred_frame[idx, :, y_min:y_max, x_min:x_max]
    
                # Background for loss
                background_output = output[idx].clone()
                background_target = blurred_frame[idx].clone()

                background_output[:, y_min:y_max, x_min:x_max] = 0
                background_target[:, y_min:y_max, x_min:x_max] = 0

                # Compute losses
                if background_output.ndim == 3:
                    background_output = background_output.unsqueeze(1)
                if background_target.ndim == 3:
                    background_target = background_target.unsqueeze(1)
    
                lambda_pixel = 0.3
                face_loss = criterion_face(face_region_output, face_region_target) + lambda_pixel * torch.nn.functional.l1_loss(face_region_output, face_region_target)


                num_background_pixels += background_output.numel() - face_region_output.numel()  
                background_loss += 1 - criterion_background(background_output, background_target)
    
        # Normalize losses
        face_loss /= len(bboxes)
        background_loss /= max(1, num_background_pixels)

        # Total loss
        total_loss = face_loss + background_loss
    
        # Backpropagation and optimization
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # Accumulate loss for this batch
        epoch_loss += total_loss.item()
        epoch_face_loss += face_loss.item()
        epoch_background_loss += background_loss.item()
    
        # Update progress bar and print loss
        progress_bar.set_postfix(loss=total_loss.item())

    # Store average loss for this epoch
    epoch_losses.append(epoch_loss / num_batches)
    epoch_face_losses.append(epoch_face_loss / num_batches)
    epoch_background_losses.append(epoch_background_loss / num_batches)

    print(f'Epoch {epoch + 1} Completed. Average Loss: {epoch_losses[-1]}, '
          f'Face Loss: {epoch_face_losses[-1]}, Background Loss: {epoch_background_losses[-1]}, '
          f'Total Time: {total_time:.2f}s')


# Save the final model after all epochs
final_model_path = os.path.join(checkpoint_dir, "final_model.pt")
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch_losses': epoch_losses,
    'epoch_face_losses': epoch_face_losses,
    'epoch_background_losses': epoch_background_losses,
}, final_model_path)
print(f"Final model saved at {final_model_path}")
print("Training Completed!")
